# Identify SOEP Strata in LinkedIn Data

This notebook creates functions to assign SOEP cell identifiers (custom_id) to LinkedIn observations based on matching characteristics.

In [25]:
import pandas as pd
from pathlib import Path

# Import path configurations
import sys
sys.path.insert(0, str(Path.cwd().parent.parent))
from bonn_thesis.config import SOEP_DATA_BLD, MERGED_EXP_ED_BLD, MERGED_EXP_ED_SAMPLING_BLD

## Step 1: Load SOEP Aggregated Data

Load the SOEP cell structure which contains unique combinations and their custom_ids.

In [26]:
# Load SOEP aggregated data with cell structure
soep_agg_file = SOEP_DATA_BLD / "aggregated" / "soep_agg_part_15.parquet"
soep_cells = pd.read_parquet(soep_agg_file)


## Step 2: Create Cell Lookup Function

Create a lookup dictionary mapping cell characteristics to custom_id.

In [27]:
def create_cell_lookup(soep_cells_df):
    """Create lookup dictionary mapping cell characteristics to custom_id and n_obs.
    
    Args:
        soep_cells_df: DataFrame with SOEP cell structure containing unique combinations of:
            - syear, isco_3_name, education_grouped, sex_en, state_en, custom_id, n_obs
    
    Returns:
        Dictionary: {(syear, isco_3_name, education_grouped, sex_en, state_en): 
                     {'custom_id': ..., 'n_obs': ...}}
    """
    cell_lookup = {}
    
    for _, row in soep_cells_df.iterrows():
        key = (
            row['syear'],
            row['isco_3_name'],
            row['education_grouped'],
            row['sex_en'],
            row['state_en']
        )
        cell_lookup[key] = {
            'custom_id': row['custom_id'],
            'n_obs': row['n_obs']
        }
    
    return cell_lookup

# Create the lookup
cell_lookup = create_cell_lookup(soep_cells)
print(f"Created lookup with {len(cell_lookup)} cells")
print(f"\nExample entry:")
sample_key = list(cell_lookup.keys())[0]
print(f"  Key: {sample_key}")
print(f"  Value: {cell_lookup[sample_key]}")

Created lookup with 27080 cells

Example entry:
  Key: (2013.0, 'Administration Professionals', 'Bachelor degree', 'female', 'Baden-Württemberg')
  Value: {'custom_id': 'wage_soep_exp_15_0', 'n_obs': 1}


## Step 3: Assign Custom ID to LinkedIn Data

Function to assign custom_id to each LinkedIn observation based on its characteristics.

In [ ]:
def assign_custom_id_to_linkedin(linkedin_df, cell_lookup):
    """Assign custom_id and n_obs to LinkedIn observations.
    
    Args:
        linkedin_df: DataFrame with LinkedIn data containing:
            - syear, isco_3_name, education_grouped, sex_en, state_en
        cell_lookup: Dictionary mapping cell characteristics to metadata dict
    
    Returns:
        DataFrame: LinkedIn data with custom_id and soep_n_obs columns added
    """
    result_df = linkedin_df.copy()
    
    # Create cell key and lookup metadata for each observation
    metadata = result_df.apply(
        lambda row: cell_lookup.get(
            (
                row['syear'],
                row['isco_3_name'],
                row['education_grouped'],
                row['sex_en'],
                row['state_en']
            ),
            {'custom_id': None, 'n_obs': None}
        ),
        axis=1,
        result_type='expand'
    )
    
    result_df['custom_id'] = metadata['custom_id']
    result_df['soep_n_obs'] = metadata['n_obs']
    
    # Report matching statistics
    matched = result_df['custom_id'].notna().sum()
    total = len(result_df)
    print(f"Matched {matched}/{total} observations ({matched/total*100:.1f}%)")
    
    return result_df

# Test with a single LinkedIn file
test_file = MERGED_EXP_ED_BLD / "linkedin_merged_exp_ed_001.parquet"
print(f"Loading test file: {test_file.name}")
linkedin_sample = pd.read_parquet(test_file)

print(f"\nLinkedIn observations loaded: {len(linkedin_sample)}")
print(f"Columns: {list(linkedin_sample.columns)}")

Loading test file: linkedin_merged_exp_ed_001.parquet

LinkedIn observations loaded: 1314
Columns: ['exp_id', 'prof_id', 'comp_id', 'industry_id', 'job_title_id', 'job_title', 'exp_description', 'exp_company', 'industry', 'matched_city', 'matched_state', 'bland_code', 'match_method', 'experience_at_start_recalc', 'experience_at_start_ft', 'exp_start_date', 'exp_end_date', 'duration', 'date_reconstruction_method', 'is_overlapping_reference', 'gender', 'hierarchy', 'hierarchy_name', 'comp_type', 'min_size', 'max_size', 'total_size', 'isco_3_digit', 'syear', 'pgexpft', 'ed_id', 'ed_start_date', 'ed_end_date', 'education_grouped', 'isco_2_digit', 'isco_1_digit', 'isco_1_name', 'isco_2_name', 'isco_3_name', 'sex_en', 'state_en', 'state_de']


## Step 4: Test the Assignment

Test assigning custom_id to the first LinkedIn file.

In [29]:
# Assign custom_id and n_obs
linkedin_with_id = assign_custom_id_to_linkedin(linkedin_sample, cell_lookup)

# Check results
print("\nSample of LinkedIn data with custom_id and soep_n_obs:")
print(linkedin_with_id[['syear', 'isco_3_name', 'education_grouped', 'sex_en', 'state_en', 
                        'custom_id', 'soep_n_obs']].head(10))

# Check for unmatched observations
unmatched = linkedin_with_id[linkedin_with_id['custom_id'].isna()]
print(f"\n\nUnmatched observations: {len(unmatched)} ({len(unmatched)/len(linkedin_with_id)*100:.1f}%)")

if len(unmatched) > 0:
    print("\nCharacteristics of unmatched observations:")
    unmatched_chars = unmatched.groupby(['syear', 'isco_3_name', 'education_grouped', 'sex_en', 'state_en']).size().reset_index(name='count')
    print(unmatched_chars.head(20))

Matched 895/1314 observations (68.1%)

Sample of LinkedIn data with custom_id and soep_n_obs:
    syear                                        isco_3_name  \
0  2017.0  Financial and Mathematical Associate Professio...   
1  2018.0  Financial and Mathematical Associate Professio...   
2  2013.0      Business Services and Administration Managers   
3  2014.0      Business Services and Administration Managers   
4  2015.0      Business Services and Administration Managers   
5  2016.0      Business Services and Administration Managers   
6  2017.0      Business Services and Administration Managers   
7  2018.0      Business Services and Administration Managers   
8  2019.0      Business Services and Administration Managers   
9  2013.0            Sales and Purchasing Agents and Brokers   

           education_grouped sex_en              state_en  \
0  Master or Doctoral degree   male                 Hesse   
1  Master or Doctoral degree   male                 Hesse   
2            Bache

In [31]:
linkedin_with_id.head(60)

,exp_id,prof_id,comp_id,industry_id,job_title_id,job_title,exp_description,exp_company,industry,matched_city,...,isco_2_digit,isco_1_digit,isco_1_name,isco_2_name,isco_3_name,sex_en,state_en,state_de,custom_id,soep_n_obs
0,9,3,9,4.0,8.0,Banking Supervisor,Supervision of the largest bank in a Southern ...,European Central Bank,Banking,Frankfurt,...,33,3,Technicians and Associate Professionals,Business and Administration Associate Professi...,Financial and Mathematical Associate Professio...,male,Hesse,Hessen,None,NaN
1,10,3,9,4.0,9.0,Adviser,Adviser in the Planning and Coordination of Su...,European Central Bank,Banking,Frankfurt,...,33,3,Technicians and Associate Professionals,Business and Administration Associate Professi...,Financial and Mathematical Associate Professio...,male,Hesse,Hessen,None,NaN
2,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
3,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
4,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
5,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
6,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
7,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
8,126,25,83,4.0,104.0,Vice President,Filialdirektor,UniCredit,Banking,Frankenthal,...,12,1,Managers,Administrative and Commercial Managers,Business Services and Administration Managers,male,Rhineland-Palatinate,Rheinland-Pfalz,None,NaN
9,473,110,252,66.0,392.0,Territory Executive,None,Philip Morris International,Tobacco,None,...,33,3,Technicians and Associate Professionals,Business and Administration Associate Professi...,Sales and Purchasing Agents and Brokers,male,Bavaria,Bayern,wage_soep_exp_15_3193,7.0


## Step 5: Run pytask to Process All Files

To assign custom_id and soep_n_obs to all 563 LinkedIn files, run:

```bash
pytask src/bonn_thesis/sampling/task_assign_strata.py
```

This will:
- Load SOEP cell structure once
- Process all linkedin_merged_exp_ed_*.parquet files in parallel
- Save results to bld/data/merged_exp_ed_sampling/linkedin_with_strata_*.parquet